# Semana 10: Modelos Pre-entrenados: BERT y GPT – Fine-tuning
## Notebook de Ejercicios (NB2) – Fine-tuning de BERT para Clasificación

**Propósito:** Realizar fine-tuning de BERT para clasificación de sentimiento en el dataset IMDb y comparar su rendimiento con modelos clásicos y LSTM.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Fine-tunear BERT para clasificación binaria usando el `Trainer` de Hugging Face.
- Evaluar el modelo en test y comparar con modelos clásicos (Naive Bayes, SVM) y LSTM.
- Guardar el modelo fine-tuneado y opcionalmente subirlo al Hugging Face Hub.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Instalamos dependencias necesarias
!pip install transformers datasets evaluate scikit-learn torch

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Hugging Face
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
from datasets import load_dataset, Dataset

# Scikit-learn para modelos clásicos y métricas
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# PyTorch para LSTM
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# NLTK para tokenización
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

# Para guardar el modelo
from huggingface_hub import notebook_login

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Carga y Preprocesamiento del Dataset IMDb

Cargamos el dataset IMDb y lo preparamos para fine-tuning.

In [ ]:
# Cargar dataset IMDb
print("Cargando dataset IMDb...")
dataset = load_dataset('imdb')

# Usar subconjuntos para acelerar el entrenamiento
train_size = 2000
test_size = 500

train_dataset = dataset['train'].shuffle(seed=42).select(range(train_size))
test_dataset = dataset['test'].shuffle(seed=42).select(range(test_size))

print(f"Entrenamiento: {len(train_dataset)} ejemplos")
print(f"Prueba: {len(test_dataset)} ejemplos")
print("\nEjemplo:")
print(f"Texto: {train_dataset[0]['text'][:200]}...")
print(f"Label: {train_dataset[0]['label']}")

---
## 2. Tokenización para BERT

Cargamos el tokenizer de BERT y tokenizamos los datasets.

In [ ]:
# Cargar tokenizer de BERT
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Función de tokenización
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

# Tokenizar datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Eliminar la columna 'text' (no la necesitamos para el modelo)
tokenized_train = tokenized_train.remove_columns(['text'])
tokenized_test = tokenized_test.remove_columns(['text'])

print("Tokenización completada.")
print(f"Columnas disponibles: {tokenized_train.column_names}")

---
## 3. Carga del Modelo BERT pre-entrenado

Cargamos BERT con una cabeza de clasificación para 2 clases.

In [ ]:
# Cargar modelo
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# Verificar número de parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")

---
## 4. Configuración del Entrenamiento con Trainer

Configuramos los argumentos de entrenamiento y el Trainer.

In [ ]:
# Data collator para padding dinámico
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none'
)

# Función para calcular métricas
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Crear Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer configurado.")

---
## 5. Entrenamiento (Fine-tuning) de BERT

Ejecutamos el entrenamiento. Esto puede tomar varios minutos.

In [ ]:
# Entrenar
print("Iniciando fine-tuning de BERT...")
trainer.train()

---
## 6. Evaluación del Modelo Fine-tuneado

Evaluamos el modelo en el conjunto de test.

In [ ]:
# Evaluar
eval_results = trainer.evaluate()
print("=== RESULTADOS BERT ===")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

---
## 7. Comparación con Modelos Clásicos

Entrenamos modelos clásicos (Naive Bayes, SVM) y LSTM para comparar.

In [ ]:
# Preparar datos para modelos clásicos
train_texts = train_dataset['text']
train_labels = train_dataset['label']
test_texts = test_dataset['text']
test_labels = test_dataset['label']

# Vectorizador TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_test_tfidf = tfidf_vectorizer.transform(test_texts)

# Naive Bayes
print("Entrenando Naive Bayes...")
start_nb = time.time()
nb = MultinomialNB(alpha=1.0)
nb.fit(X_train_tfidf, train_labels)
nb_time = time.time() - start_nb
nb_pred = nb.predict(X_test_tfidf)
nb_acc = accuracy_score(test_labels, nb_pred)

# SVM Lineal
print("Entrenando SVM Lineal...")
start_svm = time.time()
svm = LinearSVC(random_state=42, C=1.0, max_iter=2000)
svm.fit(X_train_tfidf, train_labels)
svm_time = time.time() - start_svm
svm_pred = svm.predict(X_test_tfidf)
svm_acc = accuracy_score(test_labels, svm_pred)

print(f"\nNaive Bayes - Accuracy: {nb_acc:.4f}, Tiempo: {nb_time:.2f}s")
print(f"SVM - Accuracy: {svm_acc:.4f}, Tiempo: {svm_time:.2f}s")

### 7.1. Modelo LSTM

Implementamos un modelo LSTM simple para comparar.

In [ ]:
# Construir vocabulario para LSTM
def build_vocab(texts, max_size=10000, min_freq=2):
    word_counts = Counter()
    for text in texts:
        word_counts.update(text.lower().split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in word_counts.items():
        if count >= min_freq and len(vocab) < max_size:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_texts)

def encode_texts(texts, vocab, max_len=200):
    encoded = []
    for text in texts:
        tokens = text.lower().split()[:max_len]
        indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        indices += [vocab['<PAD>']] * (max_len - len(indices))
        encoded.append(indices)
    return torch.tensor(encoded)

max_len = 200
X_train_lstm = encode_texts(train_texts, vocab, max_len)
X_test_lstm = encode_texts(test_texts, vocab, max_len)
y_train_lstm = torch.tensor(train_labels)
y_test_lstm = torch.tensor(test_labels)

train_loader = DataLoader(TensorDataset(X_train_lstm, y_train_lstm), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_lstm, y_test_lstm), batch_size=32, shuffle=False)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, num_layers=2, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        last_hidden = hidden[-1]
        output = self.fc(self.dropout(last_hidden))
        return output

model_lstm = LSTMClassifier(len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

def train_lstm(model, train_loader, test_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    
    # Evaluación final
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    return correct / total

print("\nEntrenando LSTM...")
lstm_acc = train_lstm(model_lstm, train_loader, test_loader, epochs=5)
print(f"LSTM - Accuracy: {lstm_acc:.4f}")

---
## 8. Comparación de Resultados

Comparamos todos los modelos.

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['Naive Bayes', 'SVM', 'LSTM', 'BERT (fine-tuned)'],
    'Accuracy': [nb_acc, svm_acc, lstm_acc, eval_results['eval_accuracy']]
})

print("=== COMPARACIÓN DE MODELOS ===")
comparacion.round(4)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(comparacion['Modelo'], comparacion['Accuracy'])
plt.xlabel('Modelo')
plt.ylabel('Accuracy')
plt.title('Comparación de Accuracy entre Modelos')
plt.ylim(0.7, 0.95)
for i, v in enumerate(comparacion['Accuracy']):
    plt.text(i, v + 0.005, f'{v:.4f}', ha='center')
plt.show()

---
## 9. Guardar el Modelo Fine-tuneado

Guardamos el modelo y el tokenizer para uso futuro.

In [ ]:
# Guardar localmente
model_save_path = './bert-imdb-sentiment'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Modelo guardado en {model_save_path}")

### 9.1. (Opcional) Subir al Hugging Face Hub

Para subir el modelo al Hub, necesitas una cuenta en Hugging Face y ejecutar `notebook_login()`.

In [ ]:
# Descomentar para iniciar sesión en Hugging Face Hub
# notebook_login()

# Subir el modelo
# model.push_to_hub('tu-usuario/bert-imdb-sentiment')
# tokenizer.push_to_hub('tu-usuario/bert-imdb-sentiment')
print("Para subir al Hub, descomenta las líneas anteriores y ejecuta.")

---
## 10. Prueba del Modelo Guardado

Cargamos el modelo guardado y probamos con algunas frases.

In [ ]:
# Cargar modelo guardado
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_save_path).to(device)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_save_path)

# Función de predicción
def predict_sentiment(text):
    inputs = loaded_tokenizer(text, return_tensors='pt', truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = loaded_model(**inputs)
        logits = outputs.logits
        prediction = torch.argmax(logits, dim=1).item()
    return 'POSITIVO' if prediction == 1 else 'NEGATIVO'

# Probar con algunas frases
test_phrases = [
    "This movie was absolutely amazing! I loved it.",
    "Terrible film, complete waste of time.",
    "The acting was good but the plot was boring.",
    "I would recommend this to everyone."
]

print("=== PRUEBAS DEL MODELO ===\n")
for phrase in test_phrases:
    sent = predict_sentiment(phrase)
    print(f"Texto: {phrase[:50]}... -> Sentimiento: {sent}")

---
## 11. Conclusiones

En este notebook hemos realizado fine-tuning de BERT para clasificación de sentimiento:

✔️ **Fine-tuning con Trainer**: Configuramos y entrenamos BERT usando la API de alto nivel de Hugging Face.

✔️ **Evaluación**: BERT supera significativamente a los modelos clásicos y LSTM (accuracy > 90%).

✔️ **Comparación**:
- Naive Bayes y SVM: ~85-88% accuracy.
- LSTM: ~87% accuracy.
- BERT fine-tuned: >90% accuracy.

✔️ **Guardado del modelo**: Guardamos el modelo localmente y opcionalmente podemos subirlo al Hub.

**Lección clave**: Los modelos pre-entrenados como BERT, con fine-tuning, superan ampliamente a los enfoques clásicos, especialmente cuando se dispone de datos moderados. El ecosistema de Hugging Face facilita enormemente este proceso.

---
**Fin del Notebook de Ejercicios - Semana 10 (NLP)**